# Introduction #

In this document the procedure used to get the dataset will be explained in detail

## FIRMS data recollection ##

First, we started by downloading historical thermal anomaly records. FIRMS offers them indexed by sensor, then year and finally country. Here is the website for that: https://firms.modaps.eosdis.nasa.gov/country/

So as to show their format, below is the example of the MODIS sensor 2020 anomalies for Uruguay:

In [47]:
import pandas as pd

# Load the MODIS wildfire data
csv_path = "data/modis_2020_Uruguay.csv"
df = pd.read_csv(csv_path)
df.head(10)

,latitude,longitude,brightness,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_t31,frp,daynight,type
0,-34.5301,-56.2798,311.7,1.2,1.1,2020-01-02,1728,Aqua,MODIS,41,6.03,296.6,4.7,D,2
1,-34.2796,-57.1021,320.6,1.4,1.2,2020-01-02,1728,Aqua,MODIS,22,6.03,302.5,18.0,D,0
2,-34.2690,-57.1038,335.0,1.4,1.2,2020-01-02,1728,Aqua,MODIS,87,6.03,304.8,44.3,D,0
3,-32.3717,-54.1746,316.3,1.1,1.0,2020-01-03,1354,Terra,MODIS,32,6.03,299.0,10.3,D,2
4,-33.4494,-58.0852,314.8,1.0,1.0,2020-01-03,1355,Terra,MODIS,35,6.03,302.1,5.1,D,0
5,-31.9398,-54.6094,339.3,3.0,1.6,2020-01-06,1425,Terra,MODIS,78,6.03,297.8,165.7,D,0
6,-31.9422,-54.5777,318.8,3.0,1.6,2020-01-06,1425,Terra,MODIS,48,6.03,295.7,42.3,D,0
7,-31.9391,-54.6162,329.7,3.0,1.6,2020-01-06,1425,Terra,MODIS,72,6.03,297.7,100.3,D,0
8,-31.9413,-54.5845,325.2,3.0,1.6,2020-01-06,1425,Terra,MODIS,75,6.03,296.1,76.4,D,0
9,-34.5801,-56.2536,328.3,1.0,1.0,2020-01-07,1747,Aqua,MODIS,74,6.03,310.4,12.7,D,0


From this data, the important ones are:
- latitude
- longitude
- acq_date
- acq_time

Multiple of these files are downloaded, and then merged into a new csv that contains all the anomalies, over which it is possible to iterate.
Then, a GEE query is performed over these elements to perform a query. A Sentinel-2 image was looked for, near the time and place of the anomaly, to check for a possible fire. Almost 100,000 images were downladed and manually reviewed, searching for fires.
Sometimes FIRMS creates more than one alert for the same fire, in some cases the exact same image is duplicated, and in some others there is a little variation in time/location.
After registering the data of the images that corresponded to fires and a process of deduplication, the following dataset was created. Some auxiliary fields like Country were added

In [48]:
import pandas as pd

# Load the MODIS wildfire data
csv_path = "data/firms_features_merged_last_matched.csv"
df = pd.read_csv(csv_path)
df.head(10)

,latitude,longitude,FIRMS_date,image_date,date_diff_hours,cloud_pct,thumbnail_file,satellite_image_source,detecion_source,country,firms_sensor
0,-16.5626,-63.2004,2024-12-30T06:37:00,2024-12-30T14:37:23.935000,8.01,11.844788,point_114.png,sentinel-2,modis_Bolivia,Bolivia,modis
1,-13.7907,-66.8908,2024-12-28T14:31:00,2024-12-28T14:46:47.082000,0.26,23.937154,point_214.png,sentinel-2,modis_Bolivia,Bolivia,modis
2,-13.7977,-66.8664,2024-12-28T06:55:00,2024-12-28T14:46:47.082000,7.86,23.937154,point_221.png,sentinel-2,modis_Bolivia,Bolivia,modis
3,-13.7851,-66.8864,2024-12-28T06:55:00,2024-12-28T14:46:47.082000,7.86,23.937154,point_222.png,sentinel-2,modis_Bolivia,Bolivia,modis
4,-16.5489,-60.6361,2024-12-27T13:52:00,2024-12-27T14:27:19.177000,0.59,0.015871,point_299.png,sentinel-2,modis_Bolivia,Bolivia,modis
5,-14.5097,-65.0704,2024-11-30T13:56:00,2024-11-30T14:36:46.960000,0.68,5.349525,point_953.png,sentinel-2,modis_Bolivia,Bolivia,modis
6,-13.7041,-65.8274,2024-11-13T14:05:00,2024-11-13T14:46:38.236000,0.69,45.378092,point_2436.png,sentinel-2,modis_Bolivia,Bolivia,modis
7,-13.7126,-65.8403,2024-11-13T14:05:00,2024-11-13T14:46:38.236000,0.69,45.378092,point_2437.png,sentinel-2,modis_Bolivia,Bolivia,modis
8,-15.8878,-64.0255,2024-10-31T13:38:00,2024-10-31T14:37:11.343000,0.99,2.175985,point_3909.png,sentinel-2,modis_Bolivia,Bolivia,modis
9,-15.8863,-64.0355,2024-10-31T13:38:00,2024-10-31T14:37:11.343000,0.99,2.175985,point_3910.png,sentinel-2,modis_Bolivia,Bolivia,modis


## Recreating Dataset ##
Below is a code that can be used to download the images in the dataset

In [49]:
import os
import pandas as pd
from pathlib import Path
import ee
from google.cloud import storage


def load_env_file(path: str) -> dict:
    env_values = {}
    if not os.path.exists(path):
        return env_values
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            env_values[key.strip()] = value.strip().strip('"').strip("'")
    return env_values


ENV_PATH = Path("config/.env")
env_values = load_env_file(ENV_PATH)
os.environ.update(env_values)

BANDS_TO_EXTRACT = ["B4", "B3", "B2"]#, "B8", "B11", "B12"]
CSV_PATH = "data/firms_features_merged_last_matched.csv"
OUTPUT_DIR = "fire_downloads"
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "downloaded_images.csv")
BUCKET_NAME = os.environ.get("BUCKET_NAME", "fire_dataset_2")
GEE_PROJECT = os.environ.get("GEE_PROJECT", "fire-detection-uruguay")

"""os.makedirs(OUTPUT_DIR, exist_ok=True)

try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Initialize()

storage_client = storage.Client()
_ = storage_client.bucket(BUCKET_NAME)

records = []

for idx, row in df.iterrows():
    lat = row["latitude"]
    lon = row["longitude"]
    image_date = row["image_date"]
    thumbnail_file = row["thumbnail_file"]
    country = str(row["country"]).strip()

    target_dt = pd.to_datetime(image_date)
    start_dt = target_dt - pd.Timedelta(hours=10)
    end_dt = target_dt + pd.Timedelta(hours=10)

    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(2000)

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate(ee.Date(start_dt.to_pydatetime()), ee.Date(end_dt.to_pydatetime()))
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
        .select(BANDS_TO_EXTRACT)
        .sort("system:time_start")
    )

    count = collection.size().getInfo()
    if count == 0:
        print(f"No suitable image found for {thumbnail_file}")
        continue

    image_list = collection.toList(count)
    target_ms = int(target_dt.timestamp() * 1000)
    best_image = None
    best_delta = None

    for i in range(count):
        img = ee.Image(image_list.get(i))
        img_time = img.get("system:time_start").getInfo()
        delta = abs(img_time - target_ms)
        if best_delta is None or delta < best_delta:
            best_delta = delta
            best_image = img

    if best_image is None:
        continue

    try:
        image_id = best_image.get("system:index").getInfo()
    except Exception:
        image_id = best_image.id().getInfo()

    safe_country = country.replace(" ", "_")
    name = f"{safe_country}_{idx}"

    export_task = ee.batch.Export.image.toCloudStorage(
        image=best_image,
        bucket=BUCKET_NAME,
        fileNamePrefix=f"{OUTPUT_DIR}/{name}",
        region=region,
        scale=10,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True},
        maxPixels=1e13,
    )
    export_task.start()

    records.append(
        {
            "latitude": lat,
            "longitude": lon,
            "acquisition_time": target_dt.isoformat(),
            "country": country,
            "image_id": image_id,
            "name": name,
        }
    )
    print(f"Queued {name} from {image_id}")

pd.DataFrame(records).to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(records)} records to {OUTPUT_CSV}")
"""

'os.makedirs(OUTPUT_DIR, exist_ok=True)\n\ntry:\n    ee.Initialize(project=GEE_PROJECT)\nexcept Exception:\n    ee.Initialize()\n\nstorage_client = storage.Client()\n_ = storage_client.bucket(BUCKET_NAME)\n\nrecords = []\n\nfor idx, row in df.iterrows():\n    lat = row["latitude"]\n    lon = row["longitude"]\n    image_date = row["image_date"]\n    thumbnail_file = row["thumbnail_file"]\n    country = str(row["country"]).strip()\n\n    target_dt = pd.to_datetime(image_date)\n    start_dt = target_dt - pd.Timedelta(hours=10)\n    end_dt = target_dt + pd.Timedelta(hours=10)\n\n    point = ee.Geometry.Point([lon, lat])\n    region = point.buffer(2000)\n\n    collection = (\n        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")\n        .filterBounds(point)\n        .filterDate(ee.Date(start_dt.to_pydatetime()), ee.Date(end_dt.to_pydatetime()))\n        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))\n        .select(BANDS_TO_EXTRACT)\n        .sort("system:time_start")\n    )\n\n

In [50]:
import os
from pathlib import Path
import pandas as pd
import ee
import requests


def load_env_file(path: str) -> dict:
    env_values = {}
    if not os.path.exists(path):
        return env_values
    with open(path, "r", encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            env_values[key.strip()] = value.strip().strip('"').strip("'")
    return env_values


def gee_initialization(project: str | None = None) -> None:
    env_values = load_env_file(Path("config/.env"))
    os.environ.update(env_values)

    gee_project = project or os.environ.get("GEE_PROJECT")
    try:
        ee.Initialize(project=gee_project)
    except Exception:
        ee.Initialize()


def read_alert_row(df: pd.DataFrame, index: int) -> dict | None:
    df = df.head(5)  # TESTING: Limit to first 5 rows for testing purposes
    if index < 0 or index >= len(df):
        return None

    row = df.iloc[index]
    target_dt = pd.to_datetime(row["image_date"])

    return {
        "lat": float(row["latitude"]),
        "lon": float(row["longitude"]),
        "datetime": target_dt,
        "country": str(row["country"]).strip(),
        "thumbnail_file": row["thumbnail_file"],
    }


def gee_get_image(lat: float, lon: float, target_dt: pd.Timestamp):
    print(f"BANDAS A SACAR SON {BANDS_TO_EXTRACT}")

    start_dt = target_dt - pd.Timedelta(hours=10)
    end_dt = target_dt + pd.Timedelta(hours=10)

    point = ee.Geometry.Point([lon, lat])
    region = point.buffer(2000)

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate(ee.Date(start_dt.to_pydatetime()), ee.Date(end_dt.to_pydatetime()))
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 80))
        .select(BANDS_TO_EXTRACT)
        .sort("system:time_start")
    )

    count = collection.size().getInfo()
    if count == 0:
        return None, None

    image_list = collection.toList(count)
    target_ms = int(target_dt.timestamp() * 1000)
    best_image = None
    best_delta = None

    for i in range(count):
        img = ee.Image(image_list.get(i))
        img_time = img.get("system:time_start").getInfo()
        delta = abs(img_time - target_ms)
        if best_delta is None or delta < best_delta:
            best_delta = delta
            best_image = img

    return best_image, region


def gee_download_image(best_image, name: str, output_dir: str, region, scale: int = 10) -> None:
    task = ee.batch.Export.image.toDrive(
        image=best_image,
        description=name,
        folder=output_dir,
        region=region,
        scale=scale,
        fileFormat="GeoTIFF",
        formatOptions={"cloudOptimized": True},
        maxPixels=1e13,
    )
    task.start()


def gee_download_local(point, image, output_filename: str, scale: int = 10) -> bool:
    region = point.buffer(2000).bounds()

    download_params = {
        "region": region,
        "scale": scale,
        "format": "GEO_TIFF",
    }

    try:
        url = image.clip(region).getDownloadURL(download_params)
        response = requests.get(url, timeout=60)

        if response.status_code != 200:
            print(f"[ERROR] HTTP {response.status_code}")
            return False

        if "image/tiff" not in response.headers.get("Content-Type", ""):
            print("[ERROR] Not a TIFF response")
            return False

        if len(response.content) < 1000:
            print("[ERROR] File too small — likely invalid")
            return False

        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with open(output_filename, "wb") as f:
            f.write(response.content)

        return True

    except Exception as e:
        print(f"[ERROR] {e}")
        return False


def gee_download_thumb_local(point, image, output_filename: str, width: int = 1024, height: int = 1024) -> bool:
    region = point.buffer(2000).bounds()

    thumb_params = {
        "region": region,
        "dimensions": f"{width}x{height}",
        "format": "png",
        "bands": ["B4", "B3", "B2"],
        "min": 0,
        "max": 3000,
        "gamma": 1.2,
    }

    try:
        url = image.clip(region).getThumbURL(thumb_params)
        response = requests.get(url, timeout=60)

        if response.status_code != 200:
            print(f"[ERROR] HTTP {response.status_code}")
            return False

        if "image/png" not in response.headers.get("Content-Type", ""):
            print("[ERROR] Not a PNG response")
            return False

        if len(response.content) < 1000:
            print("[ERROR] File too small — likely invalid")
            return False

        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with open(output_filename, "wb") as f:
            f.write(response.content)

        return True

    except Exception as e:
        print(f"[ERROR] {e}")
        return False


def save_image_info(records: list[dict], lat: float, lon: float, target_dt: pd.Timestamp, country: str, image_id: str, name: str) -> list[dict]:
    records.append(
        {
            "latitude": lat,
            "longitude": lon,
            "acquisition_time": target_dt.isoformat(),
            "country": country,
            "image_id": image_id,
            "name": name,
        }
    )
    return records

In [ ]:
from concurrent.futures import ThreadPoolExecutor

# Example workflow
gee_initialization()

DOWNLOAD_DIR = "fire_downloads"
PNG_DOWNLOAD_DIR = "fire_downloads_png"
GET_PNG = True
NUM_WORKERS = 4

os.makedirs(DOWNLOAD_DIR, exist_ok=True)

csv_path = "data/firms_features_merged_last_matched.csv"
df = pd.read_csv(csv_path)


def process_alert_row(idx: int):
    alert = read_alert_row(df, idx)
    if alert is None:
        return None

    best_image, _ = gee_get_image(alert["lat"], alert["lon"], alert["datetime"])
    if best_image is None:
        print(f"No suitable image found for row {idx}")
        return None

    try:
        image_id = best_image.get("system:index").getInfo()
    except Exception:
        image_id = best_image.id().getInfo()

    safe_country = alert["country"].replace(" ", "_")
    name = f"{safe_country}_{idx}"
    output_filename = os.path.join(DOWNLOAD_DIR, f"{name}.tif")
    thumb_filename = os.path.join(PNG_DOWNLOAD_DIR, f"{name}.png")

    point = ee.Geometry.Point([alert["lon"], alert["lat"]])
    success_tif = gee_download_local(point, best_image, output_filename)

    if success_tif:
        print(f"Downloaded {output_filename}")
    else:
        print(f"Failed to download {output_filename}")

    if GET_PNG:
        os.makedirs(PNG_DOWNLOAD_DIR, exist_ok=True)
        success_png = gee_download_thumb_local(point, best_image, thumb_filename)
        if success_png:
            print(f"Downloaded {thumb_filename}")
        else:
            print(f"Failed to download {thumb_filename}")
    else:
        print("Skipping PNG download")

    print(f"Queued {name} from {image_id}")
    return {
        "latitude": alert["lat"],
        "longitude": alert["lon"],
        "acquisition_time": alert["datetime"].isoformat(),
        "country": alert["country"],
        "image_id": image_id,
        "name": name,
    }


with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
    records = list(executor.map(process_alert_row, range(len(df))))

records = [result for result in records if result is not None]

output_csv = os.path.join(DOWNLOAD_DIR, "downloaded_images.csv")
pd.DataFrame(records).to_csv(output_csv, index=False)
print(f"Saved {len(records)} records to {output_csv}")

KeyboardInterrupt: 